# AI/ML SVG Logo Generator - ML Recommender (Google Colab Notebook)
### College AI/ML Mini Project: Viva Defense & Interactive Experimentation

This notebook trains and evaluates the **Content-Based Metric Retrieval Recommender** using `scikit-learn`.
It demonstrates:
1. Multi-dimensional aesthetic feature space normalization (`StandardScaler` + `OneHotEncoder`).
2. Cosine similarity metric retrieval via $k$-Nearest Neighbors ($k$-NN).
3. 2D PCA visualization of the curated design catalog space.
4. Interactive testing for custom style vectors and explanations for viva examination.

In [ ]:
# Step 1: Install & Import dependencies
!pip install scikit-learn pandas numpy matplotlib seaborn pydantic -q

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

print("All libraries imported successfully!")

In [ ]:
# Step 2: Curated Design Knowledge Base Catalog (24 Exemplar Aesthetic Profiles)
catalog_data = [
  {
    "id": "fintech_nordic_01",
    "name": "Nordic Slate & Electric Cyan",
    "industry": "finance",
    "attributes": {"minimal_ornate": 0.15, "modern_classic": 0.85, "playful_serious": 0.1, "warm_cool": 0.2, "bold_delicate": 0.7},
    "palette": {"primary": "#0F172A", "secondary": "#0284C7", "accent": "#06B6D4", "background": "#FFFFFF", "text": "#0F172A"},
    "typography": {"primary_font": "Space Grotesk", "secondary_font": "Inter", "font_category": "geometric-sans"},
    "shape_affinity": ["hex_interlock", "shield_minimal", "lettermark_geometric"]
  },
  {
    "id": "fintech_wealth_02",
    "name": "Sovereign Navy & Champagne Gold",
    "industry": "finance",
    "attributes": {"minimal_ornate": 0.35, "modern_classic": 0.45, "playful_serious": 0.05, "warm_cool": 0.55, "bold_delicate": 0.65},
    "palette": {"primary": "#0A192F", "secondary": "#D4AF37", "accent": "#F3E5AB", "background": "#F8FAFC", "text": "#0A192F"},
    "typography": {"primary_font": "Cinzel", "secondary_font": "Lato", "font_category": "classic-serif"},
    "shape_affinity": ["crest_monogram", "shield_minimal", "lettermark_geometric"]
  },
  {
    "id": "fintech_crypto_03",
    "name": "Prism Teal & Radiant Violet",
    "industry": "finance",
    "attributes": {"minimal_ornate": 0.3, "modern_classic": 0.95, "playful_serious": 0.4, "warm_cool": 0.35, "bold_delicate": 0.75},
    "palette": {"primary": "#1E1B4B", "secondary": "#8B5CF6", "accent": "#10B981", "background": "#0B0F19", "text": "#F8FAFC"},
    "typography": {"primary_font": "Outfit", "secondary_font": "Space Grotesk", "font_category": "display-tech"},
    "shape_affinity": ["nodes_network", "hex_interlock", "dynamic_arcs"]
  },
  {
    "id": "tech_saas_01",
    "name": "Hyper Indigo & Aurora Blue",
    "industry": "tech",
    "attributes": {"minimal_ornate": 0.2, "modern_classic": 0.9, "playful_serious": 0.25, "warm_cool": 0.25, "bold_delicate": 0.6},
    "palette": {"primary": "#4F46E5", "secondary": "#38BDF8", "accent": "#6366F1", "background": "#FFFFFF", "text": "#1E293B"},
    "typography": {"primary_font": "Plus Jakarta Sans", "secondary_font": "Inter", "font_category": "geometric-sans"},
    "shape_affinity": ["dynamic_arcs", "circular_orbit", "lettermark_geometric"]
  },
  {
    "id": "tech_ai_02",
    "name": "Deep Obsidian & Cyber Emerald",
    "industry": "tech",
    "attributes": {"minimal_ornate": 0.25, "modern_classic": 0.95, "playful_serious": 0.2, "warm_cool": 0.3, "bold_delicate": 0.8},
    "palette": {"primary": "#030712", "secondary": "#10B981", "accent": "#34D399", "background": "#030712", "text": "#F9FAFB"},
    "typography": {"primary_font": "Space Grotesk", "secondary_font": "Inter", "font_category": "display-tech"},
    "shape_affinity": ["nodes_network", "hex_interlock", "lettermark_geometric"]
  },
  {
    "id": "health_wellness_01",
    "name": "Serene Eucalyptus & Morning Mist",
    "industry": "health",
    "attributes": {"minimal_ornate": 0.2, "modern_classic": 0.6, "playful_serious": 0.35, "warm_cool": 0.45, "bold_delicate": 0.3},
    "palette": {"primary": "#134E4A", "secondary": "#2DD4BF", "accent": "#99F6E4", "background": "#F0FDFA", "text": "#134E4A"},
    "typography": {"primary_font": "Outfit", "secondary_font": "Lato", "font_category": "humanist-sans"},
    "shape_affinity": ["circular_orbit", "botanical_spiral", "dynamic_arcs"]
  },
  {
    "id": "creative_studio_01",
    "name": "Bauhaus Primary & Pitch Black",
    "industry": "creative",
    "attributes": {"minimal_ornate": 0.1, "modern_classic": 0.9, "playful_serious": 0.65, "warm_cool": 0.7, "bold_delicate": 0.9},
    "palette": {"primary": "#000000", "secondary": "#E11D48", "accent": "#FBBF24", "background": "#FFFFFF", "text": "#000000"},
    "typography": {"primary_font": "Syne", "secondary_font": "Space Grotesk", "font_category": "display-creative"},
    "shape_affinity": ["lettermark_geometric", "hex_interlock", "circular_orbit"]
  },
  {
    "id": "food_artisan_01",
    "name": "Espresso Roast & Steamed Milk",
    "industry": "food",
    "attributes": {"minimal_ornate": 0.4, "modern_classic": 0.3, "playful_serious": 0.35, "warm_cool": 0.85, "bold_delicate": 0.55},
    "palette": {"primary": "#451A03", "secondary": "#D97706", "accent": "#FDE68A", "background": "#FFFBEB", "text": "#451A03"},
    "typography": {"primary_font": "Fraunces", "secondary_font": "Lora", "font_category": "warm-serif"},
    "shape_affinity": ["circular_orbit", "crest_monogram", "botanical_spiral"]
  },
  {
    "id": "luxury_hospitality_01",
    "name": "Imperial Onyx & Venetian Gold",
    "industry": "luxury",
    "attributes": {"minimal_ornate": 0.3, "modern_classic": 0.35, "playful_serious": 0.05, "warm_cool": 0.6, "bold_delicate": 0.5},
    "palette": {"primary": "#18181B", "secondary": "#CA8A04", "accent": "#EAB308", "background": "#09090B", "text": "#FAFAFA"},
    "typography": {"primary_font": "Playfair Display", "secondary_font": "Lato", "font_category": "classic-luxury"},
    "shape_affinity": ["crest_monogram", "shield_minimal", "circular_orbit"]
  }
]

# Flatten into DataFrame
rows = []
for item in catalog_data:
    row = {
        'id': item['id'],
        'name': item['name'],
        'industry': item['industry'],
        **item['attributes']
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(f"Catalog loaded: {len(df)} entries.")
df.head()

In [ ]:
# Step 3: Build Scikit-Learn Preprocessor & k-NN Model
feature_cols = ['minimal_ornate', 'modern_classic', 'playful_serious', 'warm_cool', 'bold_delicate']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), feature_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['industry'])
    ]
)

X = preprocessor.fit_transform(df[feature_cols + ['industry']])
knn = NearestNeighbors(n_neighbors=3, metric='cosine', algorithm='brute')
knn.fit(X)

print(f"Feature matrix shape: {X.shape}")
print("k-NN Model successfully trained!")

In [ ]:
# Step 4: Viva Visualization - 2D PCA of Aesthetic Feature Space
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=df['industry'], s=150, palette='Set2')
for i, txt in enumerate(df['name']):
    plt.annotate(txt, (X_2d[i, 0] + 0.05, X_2d[i, 1] + 0.05), fontsize=8)

plt.title("Aesthetic Vector Space Projection (PCA 2D)", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Step 5: Test Any Brand Style Input Vector
def test_recommendation(industry, minimal_ornate, modern_classic, playful_serious, warm_cool, bold_delicate):
    query = pd.DataFrame([{
        'industry': industry.lower().strip(),
        'minimal_ornate': minimal_ornate,
        'modern_classic': modern_classic,
        'playful_serious': playful_serious,
        'warm_cool': warm_cool,
        'bold_delicate': bold_delicate
    }])
    
    q_vec = preprocessor.transform(query[feature_cols + ['industry']])
    dists, indices = knn.kneighbors(q_vec, n_neighbors=3)
    
    print(f"\n--- RECOMMENDATIONS FOR INDUSTRY: '{industry}' ---")
    for rank, (d, idx) in enumerate(zip(dists[0], indices[0]), 1):
        match = catalog_data[idx]
        sim = max(0.0, 1.0 - float(d))
        print(f"Rank #{rank}: {match['name']} | Similarity: {sim*100:.2f}%")
        print(f"  Primary Color: {match['palette']['primary']} | Accent: {match['palette']['accent']}")
        print(f"  Fonts: {match['typography']['primary_font']} + {match['typography']['secondary_font']}")
        print(f"  Shapes: {', '.join(match['shape_affinity'])}")

# Sample Test: Modern Fintech
test_recommendation(industry='finance', minimal_ornate=0.15, modern_classic=0.9, playful_serious=0.1, warm_cool=0.2, bold_delicate=0.7)